# Population comparison (MINGA populations)

This notebook compares the 2024 and 2040 synthetic populations across sampling rates (10/25/50%). The 50% population is treated as the ground truth when comparing downsampling levels.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_dark"

In [2]:
base_path = Path("../MINGA/populations")
files = sorted(base_path.glob("munich_persons_*pct.csv"))

usecols = [
    "age",
    "employed",
    "sex",
    "socioprofessional_class",
    "has_driving_license",
    "has_pt_subscription",
    "is_munich_resident",
]

sample_labels = {10: "10%", 25: "25%", 50: "50%"}


def parse_metadata(path: Path) -> tuple[int, int]:
    # munich_persons_2024_10pct.csv
    parts = path.stem.split("_")
    year = int(parts[2])
    sample_pct = int(parts[3].replace("pct", ""))
    return year, sample_pct


frames = []
for path in files:
    year, sample_pct = parse_metadata(path)
    df = pd.read_csv(path, sep=";", usecols=usecols)
    df["year"] = year
    df["sample_pct"] = sample_pct
    df["sample_label"] = sample_labels.get(sample_pct, f"{sample_pct}%")
    frames.append(df)

if not frames:
    raise FileNotFoundError(f"No population files found in {base_path}")

df_all = pd.concat(frames, ignore_index=True)

sample_order = [10, 25, 50]
year_order = [2024, 2040]

## Overview

In [3]:
counts = (
    df_all.groupby(["year", "sample_pct"])
    .size()
    .reset_index(name="n_persons")
    .sort_values(["year", "sample_pct"])
)
counts

,year,sample_pct,n_persons
0,2024,10,794842
1,2024,25,1988489
2,2024,50,3980818
3,2040,10,862284
4,2040,25,2156723
5,2040,50,4317586


In [4]:
age_bins = [0, 6, 18, 30, 45, 65, 75, 85, 120]
age_labels = ["0-5", "6-17", "18-29", "30-44", "45-64", "65-74", "75-84", "85+"]

bool_cols = ["employed", "has_driving_license", "has_pt_subscription", "is_munich_resident"]

for col in bool_cols:
    df_all[col] = df_all[col].astype(bool)

df_all["age_group"] = pd.cut(
    df_all["age"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True,
)

def get_categories(col: str):
    if col == "age_group":
        return age_labels
    if col in bool_cols:
        return [False, True]
    if col == "sex":
        return sorted(df_all[col].dropna().unique().tolist())
    if col == "socioprofessional_class":
        return sorted(df_all[col].dropna().unique().tolist())
    return sorted(df_all[col].dropna().unique().tolist())


def share_table(df: pd.DataFrame, col: str, categories: list) -> pd.DataFrame:
    counts = df[col].value_counts(dropna=False)
    counts = counts.reindex(categories, fill_value=0)
    share = counts / counts.sum()
    return pd.DataFrame({col: categories, "share": share.values})


analysis_cols = [
    "age_group",
    "sex",
    "employed",
    "has_driving_license",
    "has_pt_subscription",
    "is_munich_resident",
]

## 50% comparison (2024 vs 2040)

In [8]:
df_50 = df_all[df_all["sample_pct"] == 50].copy()
df_50["year_label"] = df_50["year"].astype(str)

age_dist = df_50.groupby(["year_label", "age_group"]).size().reset_index(name="count")
age_dist["share"] = age_dist["count"] / age_dist.groupby("year_label")["count"].transform("sum")

fig = px.line(
    age_dist,
    x="age_group",
    y="share",
    color="year_label",
    markers=True,
    title="Age distribution (50% sampling): 2024 vs 2040",
    category_orders={"age_group": age_labels, "year_label": ["2024", "2040"]},
)
fig.update_yaxes(title="Share")
fig.update_traces(line={"width": 1})
fig.show()

bool_cols_plot = [c for c in bool_cols if c in analysis_cols]

if bool_cols_plot:
    bool_rows = []
    for col in bool_cols_plot:
        dist = df_50.groupby(["year_label", col]).size().reset_index(name="count")
        dist["share"] = dist["count"] / dist.groupby("year_label")["count"].transform("sum")
        dist_true = dist[dist[col] == True]
        for _, row in dist_true.iterrows():
            bool_rows.append(
                {
                    "variable": col,
                    "year_label": row["year_label"],
                    "share_true": row["share"],
                }
            )

    bool_df = pd.DataFrame(bool_rows)
    fig = px.bar(
        bool_df,
        x="variable",
        y="share_true",
        color="year_label",
        barmode="group",
        title="Boolean variables (True share, 50% sampling): 2024 vs 2040",
        category_orders={"year_label": ["2024", "2040"], "variable": bool_cols_plot},
    )
    fig.update_yaxes(title="Share (True)")
    fig.show()

for col in [c for c in analysis_cols if c not in ("age_group", *bool_cols_plot)]:
    dist = df_50.groupby(["year_label", col]).size().reset_index(name="count")
    dist["share"] = dist["count"] / dist.groupby("year_label")["count"].transform("sum")

    fig = px.bar(
        dist,
        x=col,
        y="share",
        color="year_label",
        barmode="group",
        title=f"{col} distribution (50% sampling): 2024 vs 2040",
        category_orders={"year_label": ["2024", "2040"], col: get_categories(col)},
    )
    fig.update_yaxes(title="Share")
    fig.show()

/var/folders/m_/fjnjc1sn0ggc7z_2y7n27xfh0000gn/T/ipykernel_84062/567333546.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Agreement metrics (50% only)

Lower values indicate higher similarity.

In [9]:
def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

In [10]:
def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


df_2024 = df_all[(df_all["year"] == 2024) & (df_all["sample_pct"] == 50)]
df_2040 = df_all[(df_all["year"] == 2040) & (df_all["sample_pct"] == 50)]

rows = []
for col in analysis_cols:
    categories = get_categories(col)
    p = share_table(df_2024, col, categories)["share"].to_numpy()
    q = share_table(df_2040, col, categories)["share"].to_numpy()
    rows.append({"variable": col, "js_divergence": js_divergence(p, q)})

metrics = pd.DataFrame(rows).sort_values("js_divergence", ascending=False)

fig = px.bar(
    metrics,
    x="variable",
    y="js_divergence",
    title="Jensen-Shannon divergence by variable (50% sampling)",
)
fig.update_yaxes(title="JS divergence")
fig.show()

metrics

,variable,js_divergence
4,has_driving_license,1.387694e-03
2,employed,8.788273e-04
6,is_munich_resident,5.750564e-04
0,age_group,1.219417e-05
5,has_pt_subscription,1.047758e-05
1,sex,2.265877e-08
3,socioprofessional_class,0.000000e+00
